# DB 데이터 추출 및 분석 대상 필터링

PostgreSQL DB에서 수집된 Steam 인디게임 데이터를 `data/raw/`에 CSV로 저장한 뒤, `steamspy_indie_games.csv`를 기준으로 분석 대상 게임을 필터링하고 메인 분석 파일을 생성한다.

| 단계 | 테이블/파일 | 설명 | 저장 파일 |
|---|---|---|---|
| DB 추출 | `steam_indie_games` | 게임 메타데이터 | `steam_indie_games.csv` |
| DB 추출 | `steam_indie_tags` | SteamSpy 태그 데이터 | `steam_indie_tags.csv` |
| DB 추출 | `steam_indie_reviews` | 리뷰 원문 및 작성자 정보 | `steam_indie_reviews.csv` |
| DB 추출 | `steam_indie_review_histogram` | 월별/일별 리뷰 집계 | `steam_indie_review_histogram.csv` |
| DB 추출 | `steam_indie_review_summary` | 게임별 리뷰 요약 통계 | `steam_indie_review_summary.csv` |
| DB 추출 | `steam_app_details` | 게임 상세 정보 | `steam_app_details.csv` |
| DB 추출 | `steamspy_indie_games` | SteamSpy 기반 인디게임 모집단 | `steamspy_indie_games.csv` |
| 필터링 | `steamspy_indie_games.csv` | 2023~2025년, 리뷰 10개 이상, EA/F2P 제외 | `steam_indie_games.csv` |

> 주의: 마지막 필터링 단계에서 `steam_indie_games.csv`는 최종 분석 대상 파일로 다시 저장된다.


## 라이브러리 임포트 및 DB 연결

In [1]:
import sys
import json as _json
import pandas as pd
from pathlib import Path

import pandas as pd
import numpy as np
import ast
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path('..').resolve().parents[1] / 'src'))
from utils.db import get_connection

PROCESSED_DIR = Path('..').resolve().parents[1] / 'data' / 'raw'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

conn = get_connection()
print('DB 연결 성공')
print(f'저장 경로: {PROCESSED_DIR}')


DB 연결 성공
저장 경로: C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\raw


## 1. steam_indie_games

Steam Store API로 수집한 게임 메타데이터. 장르, 가격, 출시일, 소유자 수 등 핵심 정보를 포함한다.

In [2]:
df_games = pd.read_sql('SELECT * FROM steam_indie_games ORDER BY appid', conn)

out = PROCESSED_DIR / 'steam_indie_games.csv'
df_games.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_games: {len(df_games):,}행 → {out.name}')
print(f'컬럼: {df_games.columns.tolist()}')
df_games.head(3)

steam_indie_games: 9,692행 → steam_indie_games.csv
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'total_reviews', 'owners_lower', 'is_f2p', 'is_early_access', 'name']


,appid,owners,positive,negative,price,ccu,genres,release_date,developers,total_reviews,owners_lower,is_f2p,is_early_access,name
0,226620,"200,000 .. 500,000",1912,364,1499,12,"['Adventure', 'Casual', 'Indie', 'RPG', 'Strat...",2023-04-18,QCF Design,2276,200000,False,False,Desktop Dungeons
1,230210,"0 .. 20,000",303,45,2499,3,"['Adventure', 'Indie']",2025-03-13,Senscape,348,0,False,False,ASYLUM
2,251570,"10,000,000 .. 20,000,000",327889,42157,4499,17045,"['Action', 'Adventure', 'Indie', 'RPG', 'Simul...",2024-07-25,The Fun Pimps,370046,10000000,False,False,7 Days to Die


## 2. steam_indie_tags

SteamSpy API로 수집한 게임 태그 데이터. `tags` 컬럼은 JSONB 형식으로 저장되어 있어 문자열로 변환한다.

In [3]:
df_tags = pd.read_sql('SELECT * FROM steam_indie_tags ORDER BY appid', conn)

# tags 컬럼(JSONB) → 문자열 변환
df_tags['tags'] = df_tags['tags'].apply(
    lambda x: _json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else x
)

out = PROCESSED_DIR / 'steam_indie_tags.csv'
df_tags.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_tags: {len(df_tags):,}행 → {out.name}')
print(f'컬럼: {df_tags.columns.tolist()}')
df_tags.head(3)

steam_indie_tags: 9,706행 → steam_indie_tags.csv
컬럼: ['appid', 'name', 'developer', 'publisher', 'owners', 'positive', 'negative', 'price', 'tags']


,appid,name,developer,publisher,owners,positive,negative,price,tags
0,226620,Desktop Dungeons,QCF Design,QCF Design,"200,000 .. 500,000",1912,364,1499,"{""2D"": 36, ""RPG"": 103, ""Dwarf"": 18, ""Casual"": ..."
1,230210,ASYLUM,Senscape,Senscape,"0 .. 20,000",303,45,2499,"{""Dark"": 40, ""Gore"": 52, ""Indie"": 76, ""Gothic""..."
2,251570,7 Days to Die,The Fun Pimps,The Fun Pimps Entertainment LLC,"10,000,000 .. 20,000,000",327889,42157,4499,"{""FPS"": 3827, ""Voxel"": 4264, ""Action"": 3694, ""..."


## 3. steam_indie_reviews

수집된 전체 리뷰 데이터. 236,000건 이상으로 용량이 크므로 청크 단위로 읽어 저장한다.

In [4]:
out = PROCESSED_DIR / 'steam_indie_reviews.csv'

chunk_size = 50000
total = 0
for i, chunk in enumerate(
    pd.read_sql('SELECT * FROM steam_indie_reviews ORDER BY appid, timestamp_created', conn, chunksize=chunk_size)
):
    chunk.to_csv(out, index=False, encoding='utf-8-sig', mode='w' if i == 0 else 'a', header=(i == 0))
    total += len(chunk)
    print(f'  청크 {i+1}: {total:,}행 저장 완료')

print(f'\nsteam_indie_reviews: 총 {total:,}행 → {out.name}')

  청크 1: 50,000행 저장 완료
  청크 2: 100,000행 저장 완료
  청크 3: 150,000행 저장 완료
  청크 4: 200,000행 저장 완료
  청크 5: 236,379행 저장 완료

steam_indie_reviews: 총 236,379행 → steam_indie_reviews.csv


## 4. steam_indie_review_histogram

게임별 월별(`rollups`) 및 일별(`recent`) 리뷰 집계 데이터.

In [5]:
df_hist = pd.read_sql(
    'SELECT * FROM steam_indie_review_histogram ORDER BY appid, data_type, date', conn
)

out = PROCESSED_DIR / 'steam_indie_review_histogram.csv'
df_hist.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_histogram: {len(df_hist):,}행 → {out.name}')
print(f'컬럼: {df_hist.columns.tolist()}')
print(f'\ndata_type 분포:')
print(df_hist['data_type'].value_counts().to_string())
df_hist.head(3)

steam_indie_review_histogram: 11,782행 → steam_indie_review_histogram.csv
컬럼: ['appid', 'name', 'stratum', 'release_date', 'hist_start_date', 'hist_end_date', 'date', 'recommendations_up', 'recommendations_down', 'data_type']

data_type 분포:
data_type
recent     5976
rollups    5806


,appid,name,stratum,release_date,hist_start_date,hist_end_date,date,recommendations_up,recommendations_down,data_type
0,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-07,0,0,recent
1,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-08,0,0,recent
2,402160,Star Command Galaxies,Strategy_high,2024-08-28,2015-09-17,2026-04-05,2026-03-09,0,0,recent


## 5. steam_indie_review_summary

게임별 리뷰 요약 통계 (review_score, 긍정/부정 수 등). 수집 시점의 전체 누적 리뷰 기준이다.

In [6]:
df_summary = pd.read_sql('SELECT * FROM steam_indie_review_summary ORDER BY appid', conn)

out = PROCESSED_DIR / 'steam_indie_review_summary.csv'
df_summary.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_indie_review_summary: {len(df_summary):,}행 → {out.name}')
print(f'컬럼: {df_summary.columns.tolist()}')
df_summary.head(3)

steam_indie_review_summary: 200행 → steam_indie_review_summary.csv
컬럼: ['appid', 'review_score', 'review_score_desc', 'total_positive', 'total_negative', 'total_reviews']


,appid,review_score,review_score_desc,total_positive,total_negative,total_reviews
0,402160,5,Mixed,152,205,357
1,437440,5,Mixed,66,30,96
2,444690,5,Mixed,111,135,246


## 6. steam_app_details

Steam Store API로 수집한 게임 상세 정보 (설명, 장르, 카테고리, 스크린샷 등 전체 필드). `genres` 배열에 `Indie`가 포함된 게임만 조회한다.

In [7]:
df_app_details = pd.read_sql(
    "SELECT * FROM steam_app_details WHERE genres LIKE '%Indie%' ORDER BY appid",
    conn
)

out = PROCESSED_DIR / 'steam_app_details.csv'
df_app_details.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steam_app_details: {len(df_app_details):,}행 → {out.name}')
print(f'컬럼: {df_app_details.columns.tolist()}')
df_app_details.head(3)

steam_app_details: 116,681행 → steam_app_details.csv
컬럼: ['appid', 'name', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'developers', 'publishers', 'genres', 'categories', 'coming_soon', 'release_date', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


,appid,name,type,is_free,controller_support,short_description,supported_languages,developers,publishers,genres,...,final_formatted,windows,mac,linux,recommendations_total,metacritic_score,metacritic_url,achievements_total,header_image,website
0,1002,Rag Doll Kung Fu,game,False,NaN,A piece of Steam history - THE FIRST EVER NON ...,English,Mark Healey,Mark Healey,Indie,...,"₩ 1,100",True,False,False,NaN,69.0,https://www.metacritic.com/game/pc/rag-doll-ku...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.ragdollkungfu.com/
1,1500,Darwinia,game,False,full,"Darwinia blends real-time strategy, action, an...","English, German, French, Italian, Spanish - Spain",Introversion Software,Introversion Software,"Indie, Strategy",...,"₩ 13,500",True,True,True,773.0,84.0,https://www.metacritic.com/game/pc/darwinia?ft...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.darwinia.co.uk/
2,1510,Uplink,game,False,NaN,Uplink lets you play as a freelance hacker tac...,English,Introversion Software,Introversion Software,"Indie, Strategy",...,"₩ 13,500",True,True,True,1744.0,75.0,https://www.metacritic.com/game/pc/uplink-hack...,NaN,https://shared.akamai.steamstatic.com/store_it...,http://www.uplink.co.uk/


## 7. steamspy_indie_games

SteamSpy API 기반 인디게임 모집단 데이터. `02_filter_analysis_targets`에서 분석 대상 선별의 기준 데이터로 사용한다.


In [8]:
df_steamspy_games = pd.read_sql(
    'SELECT * FROM steamspy_indie_games ORDER BY appid',
    conn
)

out = PROCESSED_DIR / 'steamspy_indie_games.csv'
df_steamspy_games.to_csv(out, index=False, encoding='utf-8-sig')

print(f'steamspy_indie_games: {len(df_steamspy_games):,}행 → {out.name}')
print(f'컬럼: {df_steamspy_games.columns.tolist()}')
df_steamspy_games.head(3)


steamspy_indie_games: 61,266행 → steamspy_indie_games.csv
컬럼: ['appid', 'spy_name', 'owners', 'positive', 'negative', 'price_spy', 'ccu', 'name_store', 'type', 'genres', 'release_date', 'developers']


,appid,spy_name,owners,positive,negative,price_spy,ccu,name_store,type,genres,release_date,developers
0,1002,Rag Doll Kung Fu,"20,000 .. 50,000",91,30,99,0,Rag Doll Kung Fu,game,['Indie'],"12 Oct, 2005",Mark Healey
1,1500,Darwinia,"0 .. 20,000",864,216,1199,2,Darwinia,game,"['Indie', 'Strategy']","1 Dec, 2005",Introversion Software
2,1510,Uplink,"500,000 .. 1,000,000",2143,216,1199,2,Uplink,game,"['Indie', 'Strategy']","23 Aug, 2006",Introversion Software


## 8. DB 연결 종료 및 저장 결과 요약

In [9]:
conn.close()
print('DB 연결 종료')

print('=== 저장 완료 파일 목록 ===')
for f in sorted(PROCESSED_DIR.glob('*.csv')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'  {f.name:<50} {size_mb:>7.1f} MB')


DB 연결 종료
=== 저장 완료 파일 목록 ===
  steam_app_details.csv                                 78.7 MB
  steam_indie_games.csv                                  1.3 MB
  steam_indie_review_histogram.csv                       1.1 MB
  steam_indie_review_summary.csv                         0.0 MB
  steam_indie_reviews.csv                               76.8 MB
  steam_indie_tags.csv                                   4.2 MB
  steamspy_indie_games.csv                               8.4 MB


---

# 분석 대상 필터링


## 라이브러리 임포트


## 데이터 로드 및 파생 컬럼 생성

In [10]:
df = pd.read_csv(PROCESSED_DIR / 'steamspy_indie_games.csv')

df['total_reviews'] = df['positive'] + df['negative']
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')

def parse_genres(g):
    try:
        return ast.literal_eval(g)
    except Exception:
        return []

df['genres']       = df['genres'].apply(parse_genres)

## 분석 대상 필터링

In [11]:
MIN_REVIEWS = 10

is_ea  = df['genres'].apply(lambda gl: 'Early Access' in gl)
is_f2p = df['genres'].apply(lambda gl: 'Free To Play' in gl)

df_ea  = df[is_ea].copy()
df_f2p = df[~is_ea & is_f2p].copy()
df_f   = df[
    (df['total_reviews'] >= MIN_REVIEWS) &
    (df['release_date'].dt.year >= 2023) &
    (df['release_date'].dt.year <= 2025) &
    (~is_ea) &
    (~is_f2p)
].copy()

print(f'전체              : {len(df):,}개')
print(f'Early Access 제외 : {len(df_ea):,}개 → 별도 분석')
print(f'F2P 제외          : {len(df_f2p):,}개 → 별도 분석')
print(f'메인 모집단       : {len(df_f):,}개  (2023~2025년, 리뷰 {MIN_REVIEWS}개 이상, EA·F2P 제외)')
print(f'\n출시연도 분포 (메인):')
print(df_f['release_date'].dt.year.value_counts().sort_index().to_string())

전체              : 61,266개
Early Access 제외 : 6,340개 → 별도 분석
F2P 제외          : 3,064개 → 별도 분석
메인 모집단       : 9,692개  (2023~2025년, 리뷰 10개 이상, EA·F2P 제외)

출시연도 분포 (메인):
release_date
2023    3498
2024    4180
2025    2014


## 컬럼 정제

분석에 적합한 형태로 컬럼명을 정리하고 불필요한 컬럼을 제거한다.

In [12]:
# ── name_store 값으로 spy_name 대체 후 name 컬럼 생성 ────────────────────────
df_f['name'] = df_f['name_store'].fillna(df_f['spy_name'])

# ── price_spy → price 컬럼명 변경 ────────────────────────────────────────────
df_f = df_f.rename(columns={'price_spy': 'price'})

# ── release_date → yyyy-MM-dd 포맷 ───────────────────────────────────────────
df_f['release_date'] = df_f['release_date'].dt.strftime('%Y-%m-%d')

# ── 불필요 컬럼 제거 ──────────────────────────────────────────────────────────
df_f = df_f.drop(columns=['spy_name', 'name_store', 'type'])

print('전처리 완료')
print(df_f[['name', 'release_date', 'price']].head())


전처리 완료
                                   name release_date  price
564                    Desktop Dungeons   2023-04-18   1499
598                              ASYLUM   2025-03-13   2499
848                       7 Days to Die   2024-07-25   4499
868   Defender's Quest 2: Mists of Ruin   2025-01-30   1999
1178                 Secrets of Grindea   2024-02-29   1499


## steam_indie_review_summary 병합

- `total_reviews`, `positive`, `negative` → `steam_indie_review_summary` 값으로 대체
- `review_score`, `review_score_desc` 컬럼 추가


In [13]:
reviews = pd.read_csv(PROCESSED_DIR / 'steam_indie_review_summary.csv')

common_cols_reviews = sorted(set(df_f.columns) & set(reviews.columns))
print('games vs review_summary 공통 컬럼:', common_cols_reviews)

replace_cols = reviews[['appid', 'total_reviews', 'total_positive', 'total_negative']]
extra_cols = reviews.drop(columns=['total_reviews', 'total_positive', 'total_negative'])

df_f = df_f.merge(replace_cols, on='appid', how='left', suffixes=('_old', ''))
df_f['total_reviews'] = df_f['total_reviews'].combine_first(df_f['total_reviews_old'])
df_f['positive'] = df_f['total_positive'].combine_first(df_f['positive'])
df_f['negative'] = df_f['total_negative'].combine_first(df_f['negative'])
df_f = df_f.drop(columns=['total_reviews_old', 'total_positive', 'total_negative'])

df_f = df_f.merge(extra_cols, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('컬럼:', df_f.columns.tolist())


games vs review_summary 공통 컬럼: ['appid', 'total_reviews']
병합 결과 shape: (9692, 13)
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'genres', 'release_date', 'developers', 'name', 'total_reviews', 'review_score', 'review_score_desc']


## steam_app_details 병합

- 공통 컬럼(`name`, `developers`, `genres`, `release_date`) → `steam_app_details` 값으로 대체
- 나머지 `steam_app_details` 컬럼은 그대로 추가


In [14]:
appdetails = pd.read_csv(PROCESSED_DIR / 'steam_app_details.csv')

common_cols_appdetails = sorted(set(df_f.columns) & set(appdetails.columns))
print('games vs app_details 공통 컬럼:', common_cols_appdetails)

replace_cols2 = ['name', 'developers', 'genres', 'release_date']
appdetails_replace = appdetails[['appid'] + replace_cols2]
appdetails_extra = appdetails.drop(columns=replace_cols2)

df_f = df_f.merge(appdetails_replace, on='appid', how='left', suffixes=('_old', ''))
for col in replace_cols2:
    df_f[col] = df_f[col].combine_first(df_f[f'{col}_old'])
    df_f = df_f.drop(columns=[f'{col}_old'])

df_f = df_f.merge(appdetails_extra, on='appid', how='left')

print('병합 결과 shape:', df_f.shape)
print('컬럼:', df_f.columns.tolist())


games vs app_details 공통 컬럼: ['appid', 'developers', 'genres', 'name', 'release_date']
병합 결과 shape: (9692, 36)
컬럼: ['appid', 'owners', 'positive', 'negative', 'price', 'ccu', 'total_reviews', 'review_score', 'review_score_desc', 'name', 'developers', 'genres', 'release_date', 'type', 'is_free', 'controller_support', 'short_description', 'supported_languages', 'publishers', 'categories', 'coming_soon', 'currency', 'initial', 'final', 'discount_percent', 'initial_formatted', 'final_formatted', 'windows', 'mac', 'linux', 'recommendations_total', 'metacritic_score', 'metacritic_url', 'achievements_total', 'header_image', 'website']


# 데이터 저장

In [15]:
out_path = PROCESSED_DIR / 'steam_indie_games.csv'
df_f.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'저장 완료 → {out_path} ({len(df_f):,}개)')


저장 완료 → C:\Users\joon5\Documents\github\steam-indie-game-analysis\data\raw\steam_indie_games.csv (9,692개)
